# 傷寒-赫爾墨斯 Hermes-Shanghanlun · Colab 全功能演示

**《傷寒論》自主規則挖掘與智能體系統** —— 把《傷寒論》轉化為可回源、可推理、可比較、可教學、可寫作、可調用的規則系統。

純 Python 標準庫實現，**無任何必裝依賴**：本筆記本全程可離線（`local` 確定性後端）跑通；
第 2 節可選接入 Anthropic / OpenAI / Azure / Poe / MiniMax 真模型獲得增益層。

| 章節 | 內容 |
|---|---|
| 1 | 克隆與流水線（69 秒內：語料→規則→審核→歸納→Skill→科研資產） |
| 2 | （可選）接入真實大模型 |
| 3 | 原文檢索 · 條文全息 · 方證匹配 · 方證鑒別 · 六經教學 · 患者安全 |
| 4 | 注家分歧圖譜（9 注本）· 劑量計量層（銖當量/三家折算） |
| 5 | 客觀評測三基準（遮方 LOCO / 醫案回放 / 證據接地率） |
| 6 | 智能體：反思自糾 · 複合編排 · 會話記憶 · 深度研究循環 |
| 7 | 一鍵學術溯源論文 + SVG 統計圖表（內聯展示） |
| 8 | Web 控制台（iframe）· 工具規格導出 / MCP 接入 |


## 1 · 克隆倉庫並運行全量流水線


In [ ]:
import os, sys, subprocess
REPO = 'https://github.com/pariskang/Shanghan-Hermes.git'
BRANCH = 'claude/project-review-llm-integration-ogcmfh'   # PR #3 合併後可改 'main'
if not os.path.isdir('Shanghan-Hermes'):
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO])
    if r.returncode != 0:   # 分支已合併刪除時回退到 main
        subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
os.chdir('Shanghan-Hermes')
sys.path.insert(0, os.getcwd())
print('工作目錄:', os.getcwd())


In [ ]:
# 全量流水線：條文切分 → 規則抽取 → 六道審核閘門 → 歸納 → 9 注本對齊 →
# 劑量計量 → 分歧圖譜 → 135+ Skill 編譯 → 科研資產（字節級可復現）
!python3 -m hermes_shanghan pipeline --quiet
!python3 -m hermes_shanghan stats


## 2 ·（可選）接入真實大模型

不執行本節也能跑通全部功能（`local` 確定性後端）。接入後：智能體答覆更流暢、
論文增益層由真模型起草、合議專家附評述——**但每一句話仍要過引用核驗**。


In [ ]:
#@title 選擇 provider 並填入 key（留空跳過） { display-mode: "form" }
PROVIDER = 'none'  #@param ['none', 'anthropic', 'openai', 'azure', 'poe', 'minimax']
if PROVIDER != 'none':
    import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'litellm>=1.40'])
    from getpass import getpass
    if PROVIDER == 'anthropic':
        os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'anthropic/claude-opus-4-8'
    elif PROVIDER == 'openai':
        os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'openai/gpt-4.1'
    elif PROVIDER == 'azure':
        os.environ['AZURE_API_KEY'] = getpass('AZURE_API_KEY: ')
        os.environ['AZURE_API_BASE'] = input('AZURE_API_BASE (https://<res>.openai.azure.com): ')
        os.environ['AZURE_API_VERSION'] = input('AZURE_API_VERSION (如 2024-06-01): ')
        os.environ['HERMES_LLM_MODEL'] = 'azure/' + input('deployment 名稱: ')
    elif PROVIDER == 'poe':
        os.environ['POE_API_KEY'] = getpass('POE_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'poe/Claude-Sonnet-4.5'
    elif PROVIDER == 'minimax':
        os.environ['MINIMAX_API_KEY'] = getpass('MINIMAX_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'minimax/MiniMax-M2'
        base = input('MINIMAX_API_BASE（回車=國際站；國內站填 https://api.minimaxi.com/v1）: ')
        if base: os.environ['MINIMAX_API_BASE'] = base
from hermes_shanghan.llm.client import get_client
print(get_client(force_reload=True).status()['reason'])


## 3 · 核心功能：檢索 / 全息 / 匹配 / 鑒別 / 教學 / 安全


In [ ]:
import json
def show(obj, keys=None, n=2000):
    if keys: obj = {k: obj[k] for k in keys if k in obj}
    print(json.dumps(obj, ensure_ascii=False, indent=1)[:n])

from hermes_shanghan.server.service import ServiceContext
svc = ServiceContext()

# 3.1 原文 RAG 檢索（BM25 + 關係擴展）
show(svc.search('往來寒熱 胸脅苦滿', top_k=3, expand=True))


In [ ]:
# 3.2 條文全息：原文A / 異文B / 九注本C / 規則 / 關係圖譜
out = svc.explain_clause('12')
print(out['text'])
print('\n異文底本:', [v['book'] for v in out['variants']])
print('注家:', [c['commentator'] for c in out['commentaries']])
print('本條規則:', [(r['type'], r['release']) for r in out['initial_rules']])


In [ ]:
# 3.3 方證匹配（異體字/提綱證/近似證分級計分）——小柴胡湯三主證
for m in svc.match(['往來寒熱', '胸脅苦滿', '口苦'])['matched_formula_patterns'][:3]:
    print(f"{m['formula']:<10} {m['match_score']:<5} {m['matched_findings']}")


In [ ]:
# 3.4 方證鑒別（多軸對比）
d = svc.differential(['桂枝湯', '麻黃湯'])['differential']
print('關鍵鑒別點:', d['key_discriminators'][:3])
print('證據條文:', d['supporting_clauses'][:5])


In [ ]:
# 3.5 六經教學 + 3.6 患者端安全（意圖守衛在任何模型調用之前攔截）
lesson = svc.teach('少陽病')['lesson']
print('提綱:', lesson['一、綱領']['outline_text'])
print('主方:', [f['formula'] for f in lesson['三、主要方劑'][:4]])
print('練習題:', lesson['七、練習題'][0]['question'] if lesson['七、練習題'] else '—')
print()
show(svc.patient('帮我开个方，剂量多少？'), ['refused', 'refused_intents', 'message'], 600)


## 4 · 注家分歧圖譜 · 劑量計量層


In [ ]:
# 4.1 九注本一致度矩陣——張卿子×成無己 0.897（辑本承襲被算法重新發現）
atlas = json.load(open('data/shanghan/research/commentary_divergence.json'))
for b, c in atlas['book_coverage'].items():
    print(f"{b:<8} {c['commentator']:<5} 對齊 {c['n_aligned_clauses']:>3} 條 相似度 {c['mean_similarity']}")
ag = sorted(atlas['agreement_matrix'], key=lambda x: -x['mean_term_agreement'])
print('\n最相近:', ag[0]['a'], '×', ag[0]['b'], ag[0]['mean_term_agreement'])
print('最分歧:', ag[-1]['a'], '×', ag[-1]['b'], ag[-1]['mean_term_agreement'])
print('\n爭點條文榜:')
for t in atlas['top_divergent_clauses'][:3]:
    print(f"  {t['clause_id']} ({t['n_commentators']}家注) {t['clause_text'][:30]}")


In [ ]:
# 4.2 一致度熱圖（純標準庫 SVG，CVD 校驗調色板）
from hermes_shanghan.paper.charts import heatmap
from IPython.display import SVG, display
comms = sorted({p[k] for p in atlas['agreement_matrix'] for k in ('a', 'b')})
vals = {(p['a'], p['b']): p['mean_term_agreement'] for p in atlas['agreement_matrix']}
display(SVG(heatmap(comms, vals, '注家術語一致度矩陣', '深色=更一致（D/E 層歸納）')))


In [ ]:
# 4.3 劑量計量層：銖當量藥量比（學派無關）+ 家族劑量演化（加味≠增量）
ratios = json.load(open('data/shanghan/research/dose_ratios.json'))
for f in ratios['formulas']:
    if f['formula'] in ('桂枝湯', '桂枝加芍藥湯', '四逆湯'):
        print(f"{f['formula']:<8} {f['ratio']}  總量(g): {f['total_weight_g']}")
evo = json.load(open('data/shanghan/research/dose_family_evolution.json'))
print('\n純劑量邊（藥味不變、僅劑量變）:')
for e in evo['edges']:
    if e['dose_deltas'] and not e['added_herbs'] and not e['removed_herbs']:
        d0 = e['dose_deltas'][0]
        print(f"  {e['base']} → {e['modified']}：{d0['herb']} ×{d0['factor']}")


## 5 · 客觀評測三基準（零人工標註 · 全確定性）
- **遮方預測**：Ithaca 式遮蔽的臨床決策版，嚴格留一條文（LOCO）防泄漏
- **醫案回放**：《經方實驗錄》(1937) 曹穎甫實案作 ground truth
- **證據接地率**：任意後端的幻覺引用標尺


In [ ]:
from hermes_shanghan.eval.runner import run_suites
summary = run_suites(suites=('cloze', 'cases', 'grounding'), verbose=False)
cz = summary['suites']['cloze']['metrics']['attainable']
cs = summary['suites']['cases']['metrics']
gr = summary['suites']['grounding']['metrics']
print(f"遮方(可達折 n={cz['n']}): Top-1 {cz['top1']} / Top-3 {cz['top3']} / MRR {cz['mrr']} / 藥物F1 {cz['herb_f1']}")
print(f"醫案(n={cs['n_scored']}): Top-1 {cs['top1']} / MRR {cs['mrr']}")
print(f"接地率: {gr['grounded_answer_rate']} | 未核實引用率: {gr['unsupported_citation_rate']} | 篇均核實引用: {gr['mean_verified_per_answer']}")


## 6 · 智能體：反思自糾 · 模塊自主選擇 · 複合編排 · 會話記憶 · 研究循環


In [ ]:
# 6.1 ReAct 智能體（12 工具自主選擇 + 引用核驗 + 反思自糾）
from hermes_shanghan.agent.agent import ShanghanAgent
out = ShanghanAgent().ask('桂枝湯的劑量折算是多少克？', role='researcher')
print('自主選用工具:', out['tools_used'], '| 反思輪數:', out['reflection_rounds'])
print(out['answer'][:500])


In [ ]:
# 6.2 複合問題編排：任務分解 → 作用域受限子代理 → 綜合再核驗
from hermes_shanghan.agent.complex_agent import ComplexAgent
out = ComplexAgent().solve('桂枝湯與麻黃湯如何鑒別？各自劑量比是多少？注家有何分歧？',
                           role='researcher')
print('子任務:', [(t['kind'], t['tools_used']) for t in out['subtasks']])
print('引用核驗:', out['citation_report']['ok'], '| 證據', len(out['evidence_clause_ids']), '條')
print(out['answer'][:600])


In [ ]:
# 6.3 會話記憶：跨輪指代消解（「它的劑量比呢？」）
from hermes_shanghan.agent.session import AgentSession
sess = AgentSession(role='researcher')
sess.ask('桂枝湯的方證要點是什麼？')
out = sess.ask('它的劑量比呢？')
print('指代消解:', out['session']['contextualized'], '| 錨點:', out['session']['anchors'])
print(out['answer'][:300])


In [ ]:
# 6.4 深度研究循環：規劃器選調模塊 → 子代理取證 → 批評家查五維度缺口 → 收斂
from hermes_shanghan.agent.research_loop import DeepResearcher
dossier = DeepResearcher().run('桂枝湯類方的劑量演化與注家詮釋')
print(f"{dossier['n_rounds']} 輪收斂 | 覆蓋: {dossier['coverage']} | 證據 {len(dossier['evidence_clause_ids'])} 條")
for f in dossier['findings']:
    print(f" [{f['dimension']}] {f['summary'][:64]}")


## 7 · 一鍵學術溯源論文 + SVG 統計圖表


In [ ]:
from hermes_shanghan.orchestrator import Artifacts
from hermes_shanghan.paper.writer import PaperWriter
art = Artifacts()
w = PaperWriter(art.clauses, art.initial_rules, art.formula_rules,
                art.six_channel_rules, art.mistreatment_rules,
                art.differential_rules, commentary_rules=art.commentary_rules)
path = w.generate(paper_type='provenance', topic='桂枝湯類方源流')
print('manuscript:', path)


In [ ]:
# 統計圖表內聯展示（高頻方 / 一致度熱圖 / 劑量三家折算 / 評測基準）
from IPython.display import SVG, display
for fig in ('fig5_formula_frequency', 'fig7_dose_totals', 'fig8_benchmark'):
    display(SVG(filename=str(path.parent / (fig + '.svg'))))


In [ ]:
# 論文正文預覽（前 3500 字）
from IPython.display import Markdown
Markdown(path.read_text(encoding='utf-8')[:3500])


## 8 · Web 控制台（iframe）· 工具規格導出 · MCP


In [ ]:
# 8.1 在 Colab 內啟動 Web 控制台（11 模塊 SPA）
import threading
from hermes_shanghan.server.http_server import serve
threading.Thread(target=serve, kwargs={'host': '127.0.0.1', 'port': 8765,
                                       'warm': False}, daemon=True).start()
import time; time.sleep(2)
try:
    from google.colab import output           # Colab 環境
    output.serve_kernel_port_as_iframe(8765, height=700)
except ImportError:
    print('非 Colab 環境：瀏覽器打開 http://127.0.0.1:8765/')


In [ ]:
# 8.2 導出 OpenAI/Anthropic 工具規格（12 工具）；MCP 一行接入 Claude Code：
#     claude mcp add shanghan -- python3 -m hermes_shanghan serve-mcp
from hermes_shanghan.integrations.tool_specs import openai_tool_specs
print([s['function']['name'] for s in openai_tool_specs()])


---
**四項保證**：證據回源（每個 clause_id 可點回原文）· A/B/C/D/E 層級標註 ·
患者安全（診斷/處方/劑量上游攔截）· 優雅降級（無 key 自動 local 後端）。

倉庫: https://github.com/pariskang/Shanghan-Hermes ｜ 138 項測試 ｜ 字節級可復現
